In [23]:
import torch

class ChunkReshape:
    def __init__(self, chunk_divide_sizes, map_type=1, device='cpu'):
        self.chunk_divide_sizes = chunk_divide_sizes
        self.map_type = map_type
        self.device = device
    
    def split_into_chunks(self, tensor):
        batch_size, C, H, W = tensor.shape
        chunks = []
        
        if self.map_type == 1:
            C_indices, H_indices, W_indices = [
                [sum(dim // self.chunk_divide_sizes[i] for _ in range(j)) for j in range(self.chunk_divide_sizes[i] + 1)]
                for i, dim in enumerate([C, H, W])
            ]
            
            for b in range(batch_size):
                for i in range(self.chunk_divide_sizes[0]):
                    for j in range(self.chunk_divide_sizes[1]):
                        for k in range(self.chunk_divide_sizes[2]):
                            chunk = tensor[b,
                                           C_indices[i]:C_indices[i+1], 
                                           H_indices[j]:H_indices[j+1], 
                                           W_indices[k]:W_indices[k+1]].unsqueeze(0)
                            chunks.append(chunk)

        elif self.map_type == 2:
            for b in range(batch_size):
                for i in range(self.chunk_divide_sizes[0]):
                    for j in range(self.chunk_divide_sizes[1]):
                        for k in range(self.chunk_divide_sizes[2]):
                            C_stride_indices = torch.arange(i, C, self.chunk_divide_sizes[0]).to(self.device)
                            H_stride_indices = torch.arange(j, H, self.chunk_divide_sizes[1]).to(self.device)
                            W_stride_indices = torch.arange(k, W, self.chunk_divide_sizes[2]).to(self.device)

                            chunk = tensor[b].index_select(0, C_stride_indices)
                            chunk = chunk.index_select(1, H_stride_indices)
                            chunk = chunk.index_select(2, W_stride_indices).unsqueeze(0)
                            chunks.append(chunk)
        
        return chunks

    def stack_chunks_to_form_tensor(self, chunks):
        batch_size = len(chunks) // (self.chunk_divide_sizes[0] * self.chunk_divide_sizes[1] * self.chunk_divide_sizes[2])
        result = torch.cat(chunks).view(
            batch_size, self.chunk_divide_sizes[0], self.chunk_divide_sizes[1], self.chunk_divide_sizes[2], 
            *chunks[0].shape[1:])
        return result



# map type 1

In [24]:
device = "cpu"
count = 0

In [25]:
for i in range(2):
    for j in range(2):
        for k in range(2):
              chunk = torch.index_select(
              torch.index_select(
              torch.index_select(
              x
              , 1, torch.arange(0,128,1).to(device) if i==0 else torch.arange(128,256,1).to(device)).to(device)
              , 2, torch.arange(0,5,1).to(device) if j==0 else torch.arange(5,10,1).to(device)).to(device)
              , 3, torch.arange(0,5,1).to(device) if k==0 else torch.arange(5,10,1).to(device)).to(device)

              if k==0:
                 memory = chunk

              if k==1 and j==0:
                 memory2 = torch.stack((memory, chunk), dim = 1).to(device)
              if k==1 and j==1 and i==0:
                 memory3 = torch.stack((memory2, torch.stack((memory, chunk), dim = 1)), dim = 1).to(device)
              if k==1 and j==1 and i==1:
                 output_map_type_1 = torch.stack((memory3, torch.stack((memory2, torch.stack((memory, chunk), dim = 1)), dim = 1)), dim = 1).to(device)




# map type 2

In [37]:
for i in range(2):
    for j in range(2):
        for k in range(2):
            chunk = torch.index_select(
                torch.index_select(
                    torch.index_select(
                        x.to(device),
                        1,
                        torch.arange(0, 256, 2).to(device) if i == 0 else torch.arange(1, 256, 2).to(device)
                    ),
                    2,
                    torch.arange(0, 10, 2).to(device) if j == 0 else torch.arange(1, 10, 2).to(device)
                ),
                3,
                torch.arange(0, 10, 2).to(device) if k == 0 else torch.arange(1, 10, 2).to(device)
            )

            if k == 0:
                memory = chunk

            if k == 1 and j == 0:
                memory2 = torch.stack((memory, chunk), dim=1).to(device)
            if k == 1 and j == 1 and i == 0:
                memory3 = torch.stack((memory2, torch.stack((memory, chunk), dim=1)), dim=1).to(device)
            if k == 1 and j == 1 and i == 1:
                output_map_type_2 = torch.stack((memory3, torch.stack((memory2, torch.stack((memory, chunk), dim=1)), dim=1)),dim=1).to(device)

In [28]:
# Example usage
x = torch.arange(10*256*10*10).reshape(10, 256, 10, 10)
chunk_manager = ChunkReshape([2, 2, 2], map_type=1, device='cpu')
chunks = chunk_manager.split_into_chunks(x)
final_tensor_map_type1 = chunk_manager.stack_chunks_to_form_tensor(chunks)



equal maptype1

In [29]:
print('Are the results still the same as before ? ', torch.equal(output_map_type_1, final_tensor_map_type1))


Are the results still the same as before ?  True


In [14]:
# Example usage
x = torch.arange(10*256*10*10).reshape(10, 256, 10, 10)
chunk_manager = ChunkReshape([2, 2, 2], map_type=2, device='cpu')
chunks = chunk_manager.split_into_chunks(x)
final_tensor_map_type2 = chunk_manager.stack_chunks_to_form_tensor(chunks)



Final tensor shape: torch.Size([10, 2, 2, 2, 128, 5, 5])


In [38]:
print('Are the results still the same as before ? ', torch.equal(output_map_type_2, final_tensor_map_type2))


Are the results still the same as before ?  True
